# Unified Literal Translation Experiment

This notebook compares 3 conditions:
- RAW: literal translation only
- WORLDVIEW: literal + worldview context
- UI_GUIDE: literal + worldview + sentence-type guide


In [9]:
# Optional (first run only)
# %pip install -q openai pandas python-dotenv tqdm


In [1]:
import os
import re
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    from openai import OpenAI
except ImportError as e:
    raise ImportError("openai package is missing. Run the install cell first.") from e


In [2]:
# Config
BASE_DIR_OVERRIDE = os.getenv('GAME_TRANSLATION_BASE_DIR', '').strip()
if BASE_DIR_OVERRIDE:
    BASE_DIR = Path(BASE_DIR_OVERRIDE).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'game_translation_exp', cwd.parent / 'game_translation_exp']
    BASE_DIR = next((p for p in candidates if (p / 'data').exists() and (p / 'prompts').exists()), cwd)

if not (BASE_DIR / 'data').exists():
    raise FileNotFoundError(f'Could not locate game_translation_exp base dir from cwd={Path.cwd()}')

DATA_DIR = BASE_DIR / 'data'
EVAL_DIR = BASE_DIR / 'eval'
OUT_DIR = BASE_DIR / 'outputs'
PROMPT_DIR = BASE_DIR / 'prompts'

if load_dotenv is not None:
    load_dotenv(BASE_DIR.parent / '.env', override=True)

SAMPLES_PATH = DATA_DIR / 'samples.csv'
RAW_PROMPT_PATH = PROMPT_DIR / 'U_raw_literal.txt'

RUN_DATE = os.getenv('RUN_DATE', time.strftime('%Y-%m-%d'))
MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.4-mini')
DRY_RUN = os.getenv('DRY_RUN', 'false').lower() == 'true'
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 220

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not DRY_RUN and not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY is required (.env).')

RUN_DIR = OUT_DIR / f'run_{RUN_DATE}_unified'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR =', BASE_DIR)
print('RUN_DIR =', RUN_DIR)
print('MODEL =', MODEL)
print('DRY_RUN =', DRY_RUN)
print('OPENAI_API_KEY loaded =', bool(OPENAI_API_KEY))


BASE_DIR = C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp
RUN_DIR = C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp\outputs\run_2026-04-15_unified
MODEL = gpt-5.4-mini
DRY_RUN = False
OPENAI_API_KEY loaded = True


In [3]:
# Load data and prompt resources
samples_df = pd.read_csv(SAMPLES_PATH)

DEFAULT_WORLDVIEW_CONTEXT = (
    'Destiny-like sci-fantasy universe. '
    'Tone: mysterious, heroic, high-stakes, mythic-technical blend. '
    'UI text should be concise; gameplay mechanics must remain precise.'
)

WORLDVIEW_CONTEXT = DEFAULT_WORLDVIEW_CONTEXT
for pth in [BASE_DIR.parent / 'worldview_context.txt', BASE_DIR / 'worldview_context.txt']:
    if pth.exists():
        txt = pth.read_text(encoding='utf-8').strip()
        if txt:
            WORLDVIEW_CONTEXT = txt
            print('WORLDVIEW_CONTEXT loaded from', pth)
            break

UI_GUIDE = (
    '- UI: concise and scan-friendly (keep short when possible)\n'
    '- perk: preserve conditions/numbers/order precisely\n'
    '- dialogue: natural spoken tone\n'
    '- lore: immersive narrative tone with readability\n'
    '- do not add facts not present in source'
)

RAW_TEMPLATE = RAW_PROMPT_PATH.read_text(encoding='utf-8')
WORLDVIEW_TEMPLATE = (
    '당신은 영어를 한국어로 직역하는 번역기입니다.\n\n'
    '세계관 컨텍스트:\n{WORLDVIEW_CONTEXT}\n\n'
    '원문:\n{SOURCE_TEXT}\n\n'
    '규칙:\n'
    '- 세계관 톤은 반영하되 직역 기조 유지\n'
    '- 설명 추가 금지\n'
    '- 출력은 한국어만 사용(영문 알파벳 표기 금지)\n'
    '- 고유명사/스킬명/브랜드명을 포함해 가능한 모든 표현을 한국어로 번역\n'
    '- 번역문만 출력'
)
UI_GUIDE_TEMPLATE = (
    '당신은 영어를 한국어로 직역하는 번역기입니다.\n\n'
    '세계관 컨텍스트:\n{WORLDVIEW_CONTEXT}\n\n'
    '문장 유형:\n{SENTENCE_TYPE}\n\n'
    '유형 가이드:\n{UI_GUIDE}\n\n'
    '원문:\n{SOURCE_TEXT}\n\n'
    '규칙:\n'
    '- 세계관/유형 가이드를 반영하되 직역 기조 유지\n'
    '- 원문에 없는 정보/해석/설명 추가 금지\n'
    '- 출력은 한국어만 사용(영문 알파벳 표기 금지)\n'
    '- 고유명사/스킬명/브랜드명을 포함해 가능한 모든 표현을 한국어로 번역\n'
    '- 숫자, 단위, 조건, 괄호/대괄호 기호는 보존\n'
    '- 문장 유형별 우선 규칙:\n'
    '  - UI: 짧고 즉시 읽히는 라벨 형태, 군더더기 제거\n'
    '  - perk: 게임 효과 문장으로 정확하게, 조건/원인-결과 구조 유지\n'
    '  - dialogue: 자연스러운 구어체, 과도한 번역투 금지\n'
    '  - lore: 서사 톤 유지, 의미 누락 없이 가독성 확보\n'
    '- 번역문만 출력'
)

print('samples =', len(samples_df))
print(samples_df['sentence_type'].value_counts(dropna=False))


WORLDVIEW_CONTEXT loaded from C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\worldview_context.txt
samples = 80
sentence_type
UI          24
perk        24
dialogue    16
lore        16
Name: count, dtype: int64


In [4]:
def build_prompt_raw(source_text: str) -> str:
    return RAW_TEMPLATE.replace('{SOURCE_TEXT}', source_text)


def build_prompt_worldview(source_text: str) -> str:
    prompt = WORLDVIEW_TEMPLATE
    prompt = prompt.replace('{WORLDVIEW_CONTEXT}', WORLDVIEW_CONTEXT)
    prompt = prompt.replace('{SOURCE_TEXT}', source_text)
    return prompt


def build_prompt_ui_guide(source_text: str, sentence_type: str) -> str:
    prompt = UI_GUIDE_TEMPLATE
    prompt = prompt.replace('{WORLDVIEW_CONTEXT}', WORLDVIEW_CONTEXT)
    prompt = prompt.replace('{SENTENCE_TYPE}', sentence_type)
    prompt = prompt.replace('{UI_GUIDE}', UI_GUIDE)
    prompt = prompt.replace('{SOURCE_TEXT}', source_text)
    return prompt


In [5]:
def call_model(prompt: str) -> str:
    if DRY_RUN:
        return '[DRY_RUN]'

    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.responses.create(
        model=MODEL,
        input=[
            {'role': 'system', 'content': 'You are a careful game localization assistant. Output translation only.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    return re.sub(r'\s+', ' ', (resp.output_text or '').strip())


def run_condition(name: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f'Run {name}'):
        sid = r['sample_id']
        src = str(r['source_text'])
        stype = str(r.get('sentence_type', ''))

        if name == 'RAW':
            prompt = build_prompt_raw(src)
        elif name == 'WORLDVIEW':
            prompt = build_prompt_worldview(src)
        elif name == 'UI_GUIDE':
            prompt = build_prompt_ui_guide(src, stype)
        else:
            raise ValueError(f'Unknown condition: {name}')

        out = call_model(prompt)
        rows.append({'sample_id': sid, 'sentence_type': stype, 'source_text': src, 'translation': out})

        if not DRY_RUN:
            time.sleep(0.15)

    return pd.DataFrame(rows)


In [6]:
RAW_df = run_condition('RAW', samples_df)
WV_df = run_condition('WORLDVIEW', samples_df)
UI_df = run_condition('UI_GUIDE', samples_df)

(RAW_df[['sample_id','source_text','translation']]).to_csv(RUN_DIR / 'RAW_outputs.csv', index=False, encoding='utf-8')
(WV_df[['sample_id','source_text','translation']]).to_csv(RUN_DIR / 'WORLDVIEW_outputs.csv', index=False, encoding='utf-8')
(UI_df[['sample_id','source_text','translation']]).to_csv(RUN_DIR / 'UI_GUIDE_outputs.csv', index=False, encoding='utf-8')

print('Saved RAW/WORLDVIEW/UI_GUIDE outputs to', RUN_DIR)


Run UI_GUIDE: 100%|██████████| 80/80 [02:27<00:00,  1.84s/it]

Saved RAW/WORLDVIEW/UI_GUIDE outputs to C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp\outputs\run_2026-04-15_unified


In [7]:
# Build eval sheet (all-in-one merge: A/C/E + RAW/WORLDVIEW/UI_GUIDE)

def _normalize_sample_id(x):
    s = str(x).strip()
    if s.isdigit():
        return f'S{int(s):03d}'
    return s

def _prepare_df(df):
    if df is None or 'sample_id' not in df.columns:
        return None
    out = df.copy()
    out['sample_id'] = out['sample_id'].map(_normalize_sample_id)
    return out

def _pick_latest(pattern):
    cands = sorted(OUT_DIR.glob(pattern))
    return cands[-1] if cands else None

def _load_optional(preferred_path, fallback_pattern, label):
    p = preferred_path if preferred_path.exists() else _pick_latest(fallback_pattern)
    if p is None or not p.exists():
        print(f'{label}: not found')
        return None
    try:
        df = pd.read_csv(p)
        print(f'{label}: loaded from {p}')
        return _prepare_df(df)
    except Exception as e:
        print(f'{label}: failed to load {p} - {e}')
        return None

samples_df['sample_id'] = samples_df['sample_id'].map(_normalize_sample_id)

RAW_local = _prepare_df(RAW_df) if 'RAW_df' in globals() else None
WV_local = _prepare_df(WV_df) if 'WV_df' in globals() else None
UI_local = _prepare_df(UI_df) if 'UI_df' in globals() else None

if RAW_local is None:
    RAW_local = _load_optional(RUN_DIR / 'RAW_outputs.csv', 'run_*_unified/RAW_outputs.csv', 'RAW')
if WV_local is None:
    WV_local = _load_optional(RUN_DIR / 'WORLDVIEW_outputs.csv', 'run_*_unified/WORLDVIEW_outputs.csv', 'WORLDVIEW')
if UI_local is None:
    UI_local = _load_optional(RUN_DIR / 'UI_GUIDE_outputs.csv', 'run_*_unified/UI_GUIDE_outputs.csv', 'UI_GUIDE')

A_local = _load_optional(OUT_DIR / f'run_{RUN_DATE}' / 'A_outputs.csv', 'run_*/A_outputs.csv', 'A')
C_local = _load_optional(OUT_DIR / f'run_{RUN_DATE}' / 'C_outputs.csv', 'run_*/C_outputs.csv', 'C')
D_local = _load_optional(OUT_DIR / f'run_{RUN_DATE}_external' / 'E_outputs_external.csv', 'run_*_external/E_outputs_external.csv', 'D(KG)')

merged = samples_df[['sample_id','sentence_type','source_text']].copy()

if A_local is not None and 'translation' in A_local.columns:
    merged = merged.merge(A_local[['sample_id','translation']].rename(columns={'translation':'condition_A'}), on='sample_id', how='left')
else:
    merged['condition_A'] = ''

if C_local is not None and 'translation' in C_local.columns:
    merged = merged.merge(C_local[['sample_id','translation']].rename(columns={'translation':'condition_C'}), on='sample_id', how='left')
else:
    merged['condition_C'] = ''

if D_local is not None and 'translation' in D_local.columns:
    d_cols = ['sample_id', 'translation']
    if 'relation_context_external' in D_local.columns:
        d_cols.append('relation_context_external')
    merged = merged.merge(D_local[d_cols].rename(columns={'translation':'condition_D_raw'}), on='sample_id', how='left')
    if 'relation_context_external' in merged.columns:
        _ctx = merged['relation_context_external'].fillna('').astype(str)
        merged['D_kg'] = _ctx.apply(lambda x: '★' if ('No external relation context found' not in x and x.strip() != '') else '')
    else:
        merged['D_kg'] = ''
    merged['condition_D'] = merged['condition_C']
    _has_d = merged['condition_D_raw'].fillna('').astype(str).str.strip() != ''
    _kg_hit = merged['D_kg'] == '★'
    merged.loc[_has_d & _kg_hit, 'condition_D'] = merged.loc[_has_d & _kg_hit, 'condition_D_raw']
    merged = merged.drop(columns=[c for c in ['condition_D_raw', 'relation_context_external'] if c in merged.columns])
else:
    merged['D_kg'] = ''
    merged['condition_D'] = merged['condition_C']

if RAW_local is not None and 'translation' in RAW_local.columns:
    merged = merged.merge(RAW_local[['sample_id','translation']].rename(columns={'translation':'condition_raw'}), on='sample_id', how='left')
else:
    merged['condition_raw'] = ''

if WV_local is not None and 'translation' in WV_local.columns:
    merged = merged.merge(WV_local[['sample_id','translation']].rename(columns={'translation':'condition_worldview'}), on='sample_id', how='left')
else:
    merged['condition_worldview'] = ''

if UI_local is not None and 'translation' in UI_local.columns:
    merged = merged.merge(UI_local[['sample_id','translation']].rename(columns={'translation':'condition_ui_guide'}), on='sample_id', how='left')
else:
    merged['condition_ui_guide'] = ''

for col in [
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
    'D_meaning','D_term','D_consistency','D_naturalness','D_kg_fit',
    'raw_meaning','raw_term','raw_consistency','raw_naturalness','raw_ui_fit',
    'wv_meaning','wv_term','wv_consistency','wv_naturalness','wv_ui_fit',
    'ui_meaning','ui_term','ui_consistency','ui_naturalness','ui_ui_fit',
    'best','error_type','notes'
]:
    if col not in merged.columns:
        merged[col] = ''

preferred_order = [
    'sample_id','sentence_type','source_text','condition_raw','condition_worldview','condition_ui_guide',
    'condition_A','condition_C','D_kg','condition_D',
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
    'D_meaning','D_term','D_consistency','D_naturalness','D_kg_fit',
    'raw_meaning','raw_term','raw_consistency','raw_naturalness','raw_ui_fit',
    'wv_meaning','wv_term','wv_consistency','wv_naturalness','wv_ui_fit',
    'ui_meaning','ui_term','ui_consistency','ui_naturalness','ui_ui_fit',
    'best','error_type','notes'
]
ordered_existing = [c for c in preferred_order if c in merged.columns]
remaining_cols = [c for c in merged.columns if c not in ordered_existing]
merged = merged[ordered_existing + remaining_cols]

out_eval_unified = EVAL_DIR / f'eval_sheet_prefilled_UNIFIED_{RUN_DATE}.csv'
out_eval_main = EVAL_DIR / f'eval_sheet_prefilled_{RUN_DATE}.csv'
merged.to_csv(out_eval_unified, index=False, encoding='utf-8')
merged.to_csv(out_eval_main, index=False, encoding='utf-8')
print('Saved:', out_eval_unified)
print('Saved:', out_eval_main)
print('rows:', len(merged))


A: loaded from C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp\outputs\run_2026-04-13\A_outputs.csv
C: loaded from C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp\outputs\run_2026-04-13\C_outputs.csv
D(KG): loaded from C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp\outputs\run_2026-04-13_external\E_outputs_external.csv
Saved: C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp\eval\eval_sheet_prefilled_UNIFIED_2026-04-15.csv
Saved: C:\Users\LLOYDK\Desktop\Demo\LLM_Translation-pseudo_lab\game_translation_exp\eval\eval_sheet_prefilled_2026-04-15.csv
rows: 80
